# 03: Write an optional Bedrock report

This notebook sends calculated findings from Notebook 2 to a Bedrock text model. It does not ask the model to calculate metrics or read thousands of weather rows.

Run this only if your account has access to a Bedrock model. Model charges vary.

## Load the calculated findings and verify identity

In [ ]:
from pathlib import Path
import json
import os

import boto3  # AWS SDK for Python
import pandas as pd

config_path = Path("project_config.json")
findings_path = Path("outputs/calculated_findings.json")
if not config_path.exists():
    raise FileNotFoundError("Run 01_build_pipeline.ipynb first")
if not findings_path.exists():
    raise FileNotFoundError("Run 02_query_and_visualize.ipynb first")

# Notebook 1 saved AWS names; Notebook 2 saved calculated findings.
config = json.loads(config_path.read_text())
findings = json.loads(findings_path.read_text())

AWS_PROFILE = os.getenv("AWS_PROFILE") or None
session_args = {"region_name": config["region"]}
if AWS_PROFILE:
    session_args["profile_name"] = AWS_PROFILE
session = boto3.Session(**session_args)

identity = session.client("sts").get_caller_identity()
if identity["Account"] != config["account_id"]:
    raise RuntimeError("The active AWS account does not match project_config.json")

print("AWS identity")
print(f"  Account ID:    {identity['Account']}")
print(f"  Principal ARN: {identity['Arn']}")
print("\nCalculated findings")
pd.DataFrame(findings)

## Choose a Bedrock inference profile

Use an active inference profile ID or ARN for a text model that supports the Converse API. This notebook uses the following profile as its default example:

```text
us.anthropic.claude-sonnet-4-5-20250929-v1:0
```

Set `BEDROCK_MODEL_ID` to use another active inference profile.

In [ ]:
# Default example; BEDROCK_MODEL_ID can select another active inference profile.
DEFAULT_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", DEFAULT_MODEL_ID).strip()

if not MODEL_ID:
    raise ValueError("BEDROCK_MODEL_ID cannot be empty")

if not config["region"].startswith("us-") and MODEL_ID == DEFAULT_MODEL_ID:
    raise ValueError(
        "Set BEDROCK_MODEL_ID to an active inference profile available "
        "in the configured region."
    )

print(f"Using inference profile: {MODEL_ID}")

## Build a grounded prompt

The prompt limits the report to supplied values and asks the model to state the sample limitation. Review the JSON before invoking the model.

In [ ]:
# Send calculated results to the model, not the raw weather rows.
findings_json = json.dumps(findings, indent=2)
year_range = f"{min(config['years'])} through {max(config['years'])}"
city_count = len(config["stations"])
prompt = f"""
Write a short report for a student weather-data project using only the JSON below.

Requirements:
- Cite exact values from the JSON.
- Describe observed differences without assigning a cause.
- State that the sample covers {year_range} and one airport station for each of {city_count} selected cities.
- Do not calculate new values.
- Do not add facts that are absent from the JSON.
- Use a title and no more than four short paragraphs.

Calculated findings:
{findings_json}
""".strip()

print(prompt)

## Invoke Bedrock

This cell sends the calculated findings to the selected inference profile and incurs a model charge.

In [ ]:
if not MODEL_ID.strip():
    raise ValueError("Set MODEL_ID or the BEDROCK_MODEL_ID environment variable first")

runtime = session.client("bedrock-runtime")
try:
    # Converse is Bedrock's common chat API for supported models.
    response = runtime.converse(
        modelId=MODEL_ID,
        system=[{
            "text": (
                "You edit factual data reports. Use only supplied values. "
                "If a claim is not supported by the input, leave it out."
            )
        }],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 500, "temperature": 0.1},
    )
except runtime.exceptions.ValidationException as error:
    raise ValueError(
        "Set MODEL_ID to an active, Converse-compatible inference profile ID "
        "or ARN available in the configured region."
    ) from error
except runtime.exceptions.ResourceNotFoundException as error:
    raise ValueError(
        "Set MODEL_ID to an active, Converse-compatible inference profile ID "
        "or ARN available in the configured region."
    ) from error

report = "\n".join(
    block["text"]
    for block in response["output"]["message"]["content"]
    if "text" in block
).strip()
print("Generated report\n")
print(report)

In [ ]:
report_path = Path("outputs/bedrock_report.md")
report_path.write_text(report)
print(f"Saved {report_path}")
print({
    "input_tokens": response.get("usage", {}).get("inputTokens"),
    "output_tokens": response.get("usage", {}).get("outputTokens"),
    "stop_reason": response.get("stopReason"),
})

## Review before using the report

Compare every number in the report with the findings table at the top of this notebook. Remove causal claims, unsupported facts, and inflated language. Keep the calculated JSON beside the report in the repository so a reviewer can check it.

## Why this is not RAG

RAG is useful when a model needs to retrieve passages from documents. These inputs are small, calculated records, so direct grounding is simpler. A later extension could index NOAA field definitions and station documentation for questions about methodology.